In [27]:
import pandas as pd
import re
import string
import nltk
from bs4 import BeautifulSoup
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory


In [28]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to E:\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to E:\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [29]:
data = pd.read_csv("../../dataset/politik_merge.csv")
data

,Judul,Waktu,Link,Content,tag1,tag2,tag3,tag4,tag5,source
0,Jokowi Kenakan Pakaian Adat Betawi di Sidang T...,16/08/2024,https://nasional.kompas.com/read/2024/08/16/11...,"JAKARTA, KOMPAS.com - Presiden Joko Widodo me...",Presiden Jokowi,Jokowi,sidang tahunan MPR RI 2024,Jokowi adat Betawi sidang mpr 2024,Megawati tak hadiri sidang tahunan MPR 2024,kompas
1,Amnesty International Beberkan 6 Indikator Kri...,2024-07-18,https://nasional.tempo.co/read/1893144/amnesty...,"TEMPO.CO, Jakarta - Amnesty International Indo...",Amnesty International,Amnesty International Indonesia,Kebebasan Berpendapat,Indeks Demokrasi,Revisi UU TNI,tempo
2,"Jelang Long Weekend, Stasiun Kereta Cepat Hali...","Rabu, 08 Mei 2024 19:18 WIB",https://news.detik.com/berita/d-7331666/jelang...,"Stasiun kereta cepat Whoosh di Halim, Jakarta ...",kereta cepat whoosh,stasiun halim,long weekend,NaN,NaN,detik
3,KPU Tegaskan Pemilih Tak Terdaftar di DPT Bisa...,13/02/2024,https://nasional.kompas.com/read/2024/02/13/21...,"JAKARTA, KOMPAS.com - Komisi Pemilihan Umum (...",KPU,pemilu 2024,Hasyim Asy'ari,NaN,NaN,kompas
4,Kemenag Luncurkan Gerakan Senam Haji Jaga Keta...,2024-04-29,https://nasional.tempo.co/read/1861810/kemenag...,"TEMPO.CO, Jakarta - Kementerian Agama atau Kem...",Senam Haji,Kemenag,Jemaah Haji,Ibadah Haji,Asrama Haji,tempo
...,...,...,...,...,...,...,...,...,...,...
45776,"Terima Mohammad Shtayyeh, Bamsoet: RI Tegas Du...","Minggu, 25 Agu 2024 15:24 WIB",https://news.detik.com/berita/d-7507663/terima...,Ketua MPR RI ke-16 Bambang Soesatyo menegaskan...,mpr,bamsoet,palestina,mohammad shtayyeh,NaN,detik
45777,Kegiatan Setelah Kalah Pilpres: Anies Jeda Pol...,2024-04-30,https://nasional.tempo.co/read/1862571/kegiata...,"TEMPO.CO, Jakarta - Mantan calon presiden nomo...",Anies,Ganjar,Mahfud MD,Pilpres,KAGAMA,tempo
45778,"Prabowo Bicara Malin Kundang, Hasto PDIP: Blunder",2024-01-14,https://nasional.tempo.co/read/1821140/prabowo...,"TEMPO.CO, Jakarta - Sekretaris Jenderal Partai...",PDIP,Prabowo,Malin Kundang,Hasto Kristiyanto,Jokowi,tempo
45779,"Gugatannya Disebut Cengeng dan Cacat Formil, T...",2024-03-27,https://nasional.tempo.co/read/1850093/gugatan...,"TEMPO.CO, Jakarta - Anggota dari Tim Pembela P...",Timnas Amin,Gugatan Pilpres,Prabowo-Gibran,Hotman Paris Hutapea,Otto Hasibuan,tempo


In [30]:
df = data[['Judul','Content']].head(10)
df

,Judul,Content
0,Jokowi Kenakan Pakaian Adat Betawi di Sidang T...,"JAKARTA, KOMPAS.com - Presiden Joko Widodo me..."
1,Amnesty International Beberkan 6 Indikator Kri...,"TEMPO.CO, Jakarta - Amnesty International Indo..."
2,"Jelang Long Weekend, Stasiun Kereta Cepat Hali...","Stasiun kereta cepat Whoosh di Halim, Jakarta ..."
3,KPU Tegaskan Pemilih Tak Terdaftar di DPT Bisa...,"JAKARTA, KOMPAS.com - Komisi Pemilihan Umum (..."
4,Kemenag Luncurkan Gerakan Senam Haji Jaga Keta...,"TEMPO.CO, Jakarta - Kementerian Agama atau Kem..."
5,Bawaslu Minta Jajarannya Siapkan LHP Hadapi Gu...,"TEMPO.CO, Jakarta - Anggota Badan Pengawas Pem..."
6,Polisi Ungkap Aktivitas Depoy Bandar Narkoba S...,Polisi menyebutkan pria inisial DK alias Depoy...
7,"Truk Mogok di Perlintasan Kereta Serpong, 17 P...",Satu unit truk Fuso mogok di pelintasan kereta...
8,Jalan Buntu Anies Baswedan di Pilkada Jakarta ...,"TEMPO.CO, Jakarta - Bendahara Umum PDIP, Olly ..."
9,Warga Sebut Ada Benda Serupa Jimat pada Mayat ...,"TANGERANG SELATAN, KOMPAS.com - Seutas tali b..."


In [ ]:

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

stop_factory = StopWordRemoverFactory()
stop_words = set(stop_factory.get_stop_words())


In [32]:
def preprocess_text(text):
    if pd.isnull(text):
        return []

    text = text.lower()

    text = BeautifulSoup(text, "html.parser").get_text()

    text = re.sub(r'http\S+|www\S+', '', text)

    text = text.translate(str.maketrans('', '', string.punctuation))

    text = re.sub(r'\d+', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    tokens = word_tokenize(text)

    tokens = [word for word in tokens if word not in stop_words]

    tokens = [stemmer.stem(word) for word in tokens]

    return tokens


In [33]:
df['processed'] = df['Content'].apply(preprocess_text)

df


,Judul,Content,processed
0,Jokowi Kenakan Pakaian Adat Betawi di Sidang T...,"JAKARTA, KOMPAS.com - Presiden Joko Widodo me...","[jakarta, kompascom, presiden, joko, widodo, p..."
1,Amnesty International Beberkan 6 Indikator Kri...,"TEMPO.CO, Jakarta - Amnesty International Indo...","[tempoco, jakarta, amnesty, international, ind..."
2,"Jelang Long Weekend, Stasiun Kereta Cepat Hali...","Stasiun kereta cepat Whoosh di Halim, Jakarta ...","[stasiun, kereta, cepat, whoosh, halim, jakart..."
3,KPU Tegaskan Pemilih Tak Terdaftar di DPT Bisa...,"JAKARTA, KOMPAS.com - Komisi Pemilihan Umum (...","[jakarta, kompascom, komisi, pilih, umum, kpu,..."
4,Kemenag Luncurkan Gerakan Senam Haji Jaga Keta...,"TEMPO.CO, Jakarta - Kementerian Agama atau Kem...","[tempoco, jakarta, menteri, agama, kemenag, lu..."
5,Bawaslu Minta Jajarannya Siapkan LHP Hadapi Gu...,"TEMPO.CO, Jakarta - Anggota Badan Pengawas Pem...","[tempoco, jakarta, anggota, badan, awas, pilih..."
6,Polisi Ungkap Aktivitas Depoy Bandar Narkoba S...,Polisi menyebutkan pria inisial DK alias Depoy...,"[polisi, sebut, pria, inisial, dk, alias, depo..."
7,"Truk Mogok di Perlintasan Kereta Serpong, 17 P...",Satu unit truk Fuso mogok di pelintasan kereta...,"[satu, unit, truk, fuso, mogok, lintas, kereta..."
8,Jalan Buntu Anies Baswedan di Pilkada Jakarta ...,"TEMPO.CO, Jakarta - Bendahara Umum PDIP, Olly ...","[tempoco, jakarta, bendahara, umum, pdip, olly..."
9,Warga Sebut Ada Benda Serupa Jimat pada Mayat ...,"TANGERANG SELATAN, KOMPAS.com - Seutas tali b...","[tangerang, selatan, kompascom, utas, tali, wa..."


In [34]:
df['processed_text'] = df['processed'].apply(lambda x: " ".join(x))

df


,Judul,Content,processed,processed_text
0,Jokowi Kenakan Pakaian Adat Betawi di Sidang T...,"JAKARTA, KOMPAS.com - Presiden Joko Widodo me...","[jakarta, kompascom, presiden, joko, widodo, p...",jakarta kompascom presiden joko widodo pakai b...
1,Amnesty International Beberkan 6 Indikator Kri...,"TEMPO.CO, Jakarta - Amnesty International Indo...","[tempoco, jakarta, amnesty, international, ind...",tempoco jakarta amnesty international indonesi...
2,"Jelang Long Weekend, Stasiun Kereta Cepat Hali...","Stasiun kereta cepat Whoosh di Halim, Jakarta ...","[stasiun, kereta, cepat, whoosh, halim, jakart...",stasiun kereta cepat whoosh halim jakarta timu...
3,KPU Tegaskan Pemilih Tak Terdaftar di DPT Bisa...,"JAKARTA, KOMPAS.com - Komisi Pemilihan Umum (...","[jakarta, kompascom, komisi, pilih, umum, kpu,...",jakarta kompascom komisi pilih umum kpu ri pas...
4,Kemenag Luncurkan Gerakan Senam Haji Jaga Keta...,"TEMPO.CO, Jakarta - Kementerian Agama atau Kem...","[tempoco, jakarta, menteri, agama, kemenag, lu...",tempoco jakarta menteri agama kemenag luncur s...
5,Bawaslu Minta Jajarannya Siapkan LHP Hadapi Gu...,"TEMPO.CO, Jakarta - Anggota Badan Pengawas Pem...","[tempoco, jakarta, anggota, badan, awas, pilih...",tempoco jakarta anggota badan awas pilih umum ...
6,Polisi Ungkap Aktivitas Depoy Bandar Narkoba S...,Polisi menyebutkan pria inisial DK alias Depoy...,"[polisi, sebut, pria, inisial, dk, alias, depo...",polisi sebut pria inisial dk alias depoy temu ...
7,"Truk Mogok di Perlintasan Kereta Serpong, 17 P...",Satu unit truk Fuso mogok di pelintasan kereta...,"[satu, unit, truk, fuso, mogok, lintas, kereta...",satu unit truk fuso mogok lintas kereta rel li...
8,Jalan Buntu Anies Baswedan di Pilkada Jakarta ...,"TEMPO.CO, Jakarta - Bendahara Umum PDIP, Olly ...","[tempoco, jakarta, bendahara, umum, pdip, olly...",tempoco jakarta bendahara umum pdip olly dondo...
9,Warga Sebut Ada Benda Serupa Jimat pada Mayat ...,"TANGERANG SELATAN, KOMPAS.com - Seutas tali b...","[tangerang, selatan, kompascom, utas, tali, wa...",tangerang selatan kompascom utas tali warna hi...


In [35]:
df.to_csv("../../dataset/news_preprocess.csv", index=False)

In [36]:
df['char_before'] = df['Content'].str.len()

df['char_after'] = df['processed_text'].str.len()

df['word_before'] = df['Content'].str.split().str.len()

df['word_after'] = df['processed_text'].str.split().str.len()


In [37]:
df[['char_before', 'char_after', 'word_before', 'word_after']]


,char_before,char_after,word_before,word_after
0,1677,1145,235,173
1,4906,3354,627,464
2,999,696,137,104
3,1141,681,157,107
4,2595,1754,373,274
5,1503,985,212,161
6,1969,1219,312,214
7,2915,2054,389,301
8,3517,2352,504,358
9,2645,1763,386,282
